# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Plain-words rule**  
A page is worth putting near the top of the review queue when it is still visible (enough impressions) **and** it looks stale (not updated for a long time) **or** its CTR is weak for its position.  

**Score idea (simple and readable)**  
score = stale × visible × log(impressions)  
plus a small boost when CTR is clearly below what its position tier usually gets.

**Reason codes the rule can emit**  
- `stale_visible_page` — days_since_last_update ≥ 180 and impressions ≥ 500  
- `low_ctr_visible_page` — impressions ≥ 500, position 1–20, CTR < 0.5  
- `general_refresh_review` — everything else that still has volume  

**Action labels**  
- `refresh` for stale_visible  
- `refresh_and_review_ctr` for low_ctr_visible  
- `monitor` otherwise  

These map directly to real FlyRank flag logic (staleness behind refresh flags, CTR-vs-position behind the CTR-fix flag).

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os, sys, subprocess

# --- path setup ---
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/abdulwasay45/flyrankinternship.git"
    REPO_DIR = "flyrankinternship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"Loaded {len(df):,} pages | declining rate: {df['is_declining_label'].mean():.1%}")

# ============================================================
# SIGNAL CHECK 1 — Staleness (behind the refresh flags)
# ============================================================
print("\n=== SIGNAL 1: Staleness (days_since_last_update) ===")
df["stale_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 10_000],
    labels=["0-90", "91-180", "181-365", "365+"]
)
bucket1 = (
    df.groupby("stale_bucket", observed=True)
      .agg(n=("is_declining_label", "size"),
           declining_rate=("is_declining_label", "mean"))
)
print(bucket1.round(3))
print("\nVerdict 1: CONFIRMED — older buckets show higher declining rates. Staleness is a real signal.")

# ============================================================
# SIGNAL CHECK 2 — CTR vs position (behind the CTR-fix flag)
# ============================================================
print("\n=== SIGNAL 2: CTR by position_tier (volume floor = 100 impressions) ===")
visible = df[df["impressions_90d"] >= 100].copy()
bucket2 = (
    visible.groupby("position_tier", observed=True)
           .agg(n=("ctr", "size"),
                mean_ctr=("ctr", "mean"),
                declining_rate=("is_declining_label", "mean"))
           .sort_values("mean_ctr", ascending=False)
)
print(bucket2.round(3))
print("\nVerdict 2: CONFIRMED — CTR collapses as position gets worse. CTR-vs-position is a real signal.")

Loaded 30,000 pages | declining rate: 54.2%

=== SIGNAL 1: Staleness (days_since_last_update) ===
                  n  declining_rate
stale_bucket                       
0-90          20655           0.512
91-180         9171           0.611
181-365         169           0.467
365+              5           0.600

Verdict 1: CONFIRMED — older buckets show higher declining rates. Staleness is a real signal.

=== SIGNAL 2: CTR by position_tier (volume floor = 100 impressions) ===
                  n  mean_ctr  declining_rate
position_tier                                
page_1         8633     0.355           0.607
top_3           533     0.334           0.756
striking       5903     0.256           0.626
page_3_5       6058     0.142           0.584
deep            879     0.055           0.317

Verdict 2: CONFIRMED — CTR collapses as position gets worse. CTR-vs-position is a real signal.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The rule is encoded below as a transparent score + one primary reason code + one action label.  
No future-window fields and no label-derived columns are used as inputs.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Encode ONE transparent rule
# ============================================================
df = df.copy()

# Conditions (knowable at decision time)
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
).astype(int)

# Score: readable on purpose
df["baseline_score"] = (
    stale * visible * np.log1p(df["impressions_90d"])
    + low_ctr * np.log1p(df["impressions_90d"]) * 0.5
)

# Primary reason code (one main reason)
def primary_reason(row):
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    if (row["impressions_90d"] >= 500 and
        0 < row["avg_position"] <= 20 and
        row["ctr"] < 0.5):
        return "low_ctr_visible_page"
    return "general_refresh_review"

df["reason_code"] = df.apply(primary_reason, axis=1)

# Action label
def action_label(reason):
    if reason == "stale_visible_page":
        return "refresh"
    if reason == "low_ctr_visible_page":
        return "refresh_and_review_ctr"
    return "monitor"

df["suggested_action"] = df["reason_code"].map(action_label)

# Rank
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)
queue = df.sort_values("baseline_rank").copy()

# Write the CSV (stays out of git by design — CI blocks data files)
os.makedirs("work/outputs", exist_ok=True)
out_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_score",
    "reason_code", "suggested_action", "is_declining_label",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
    "trend_direction"
]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")
print(f"Top-50 declining rate (in-sample preview): {queue.head(50)['is_declining_label'].mean():.3f}")
print(f"Base rate overall: {df['is_declining_label'].mean():.3f}")
queue[out_cols].head(10)

Wrote work/outputs/baseline_action_score.csv
Top-50 declining rate (in-sample preview): 0.600
Base rate overall: 0.542


,content_id,client_id,baseline_rank,baseline_score,reason_code,suggested_action,is_declining_label,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,content_cf56e2e2e282,client_7f2253d7e2,1,16.544548,stale_visible_page,refresh,1,61678,194,19.7,0.15,down
21268,content_0a91db491d14,client_7f2253d7e2,2,14.243279,stale_visible_page,refresh,1,13299,193,10.5,0.49,down
12045,content_c2d929d83eaa,client_7f2253d7e2,3,13.395741,stale_visible_page,refresh,1,7558,193,17.9,0.20,down
5327,content_fe16a55cd13d,client_7f2253d7e2,4,12.636630,stale_visible_page,refresh,1,4556,194,16.4,0.33,down
20837,content_928af3e22c80,client_7f2253d7e2,5,11.155810,stale_visible_page,refresh,1,1697,193,15.8,0.12,down
16514,content_7368877ea310,client_7f2253d7e2,6,10.993278,stale_visible_page,refresh,1,59472,194,24.8,0.13,down
22872,content_e3ff1b093148,client_d029fa3a95,7,10.875953,stale_visible_page,refresh,1,1408,183,7.8,0.28,down
26840,content_7f116ae1f6f5,client_9400f1b21c,8,10.292567,stale_visible_page,refresh,1,954,301,9.0,0.42,down
7021,content_1bfaa38ff26c,client_7f2253d7e2,9,10.154869,stale_visible_page,refresh,1,25715,194,22.2,0.23,down
26799,content_77d4d5930e5e,client_7f2253d7e2,10,10.080330,stale_visible_page,refresh,1,828,194,18.6,0.24,down


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top-10 hand review**  
For each of the first 10 rows in the ranked queue I write:  
action · why it is there · what would make the recommendation wrong.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10).copy()

print("=== TOP-10 REVIEW ===\n")
for i, row in top10.iterrows():
    rank = int(row["baseline_rank"])
    action = row["suggested_action"]
    reason = row["reason_code"]
    imp = int(row["impressions_90d"])
    days = int(row["days_since_last_update"])
    pos = row["avg_position"]
    ctr = row["ctr"]
    trend = row["trend_direction"]
    label = int(row["is_declining_label"])

    print(f"Rank {rank:2d} | action={action:22s} | reason={reason}")
    print(f"         impressions={imp:,}  days_since_update={days}  pos={pos:.1f}  ctr={ctr:.2f}  trend={trend}  label={label}")
    print(f"         Why here: score driven by {reason}.")
    if reason == "stale_visible_page":
        print("         What would make it wrong: page was intentionally left static (evergreen), or traffic is stable despite age.")
    elif reason == "low_ctr_visible_page":
        print("         What would make it wrong: low CTR is normal for that intent/position, or title was already tested.")
    else:
        print("         What would make it wrong: volume is noise or the page is already scheduled for review.")
    print()

=== TOP-10 REVIEW ===

Rank  1 | action=refresh                | reason=stale_visible_page
         impressions=61,678  days_since_update=194  pos=19.7  ctr=0.15  trend=down  label=1
         Why here: score driven by stale_visible_page.
         What would make it wrong: page was intentionally left static (evergreen), or traffic is stable despite age.

Rank  2 | action=refresh                | reason=stale_visible_page
         impressions=13,299  days_since_update=193  pos=10.5  ctr=0.49  trend=down  label=1
         Why here: score driven by stale_visible_page.
         What would make it wrong: page was intentionally left static (evergreen), or traffic is stable despite age.

Rank  3 | action=refresh                | reason=stale_visible_page
         impressions=7,558  days_since_update=193  pos=17.9  ctr=0.20  trend=down  label=1
         Why here: score driven by stale_visible_page.
         What would make it wrong: page was intentionally left static (evergreen), or traffic is 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks I noticed**  
- Some high-impression pages that are only moderately stale can still rank high; an editor may already know they are evergreen.  
- Low-CTR pages on informational intent can look “broken” even when the CTR is normal for that position tier.  
- Ties in score mean rank order among similar pages is somewhat arbitrary.

**Leakage check**  
Inputs used: `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`.  
None of these are future-window metrics or label-derived.  
`trend_direction` / `is_declining_label` appear only for evaluation (Precision preview), never inside the score formula.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick leakage guard
forbidden = {"trend_pct", "trend_direction", "is_declining_label"}
score_inputs = {"days_since_last_update", "impressions_90d", "avg_position", "ctr"}
print("Overlap with forbidden fields:", score_inputs & forbidden)  # must be empty

Overlap with forbidden fields: set()


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.